# New QE — one QP per Kaggle notebook

Add dataset `anginhok/enhance-for-human-mfqe-vvc-trainval-qp22-37` and enable GPU + Internet. Bootstrap automatically finds the dataset under /kaggle/input, including nested mounts. Set QP below to 22, 27, 32 or 37.

Each notebook trains one model for 300,000 successful updates, Charbonnier-only, selecting best by highest validation PSNR-Y. Validation uses the first 30 frames per test18 sequence; full test18 is excluded.


In [ ]:
%%bash
set -e
git clone https://github.com/dangdinh17/Enhance-for-Human.git /kaggle/working/Enhance-for-Human
cd /kaggle/working/Enhance-for-Human
python -m pip install -q -r requirements.txt
python -B scripts/bootstrap_kaggle.py


## Comet API key
Project: `dangdinh17/qe-for-human`. Add a Kaggle Secret named `COMET_API_KEY` and enable it for this notebook. If it is missing, this cell asks for the key privately. The key is not stored in the repository or notebook output.


In [ ]:
import os
from getpass import getpass
from kaggle_secrets import UserSecretsClient

if not os.environ.get("COMET_API_KEY"):
    try:
        key = UserSecretsClient().get_secret("COMET_API_KEY")
    except Exception:
        key = getpass("Comet API key: ")
    if not key or not key.strip():
        raise ValueError("COMET_API_KEY is empty")
    os.environ["COMET_API_KEY"] = key.strip()
    del key


## Train one QP
Change only `QP`. Running this cell again automatically resumes the matching `last.pt` if it exists. Keep the same source/config and restore the complete run directory for a later session.


In [ ]:
%%bash
set -e
QP=22
cd /kaggle/working/Enhance-for-Human
python -u scripts/train_mfqev2_window.py --config "configs/kaggle/QP${QP}.yml" --device cuda:0


## Save results
For QP22: `runs/research/newqe_kaggle_comet_charb_300k_v1/QP22/`. Other QPs have their own QP directory. Save the entire selected run directory before ending the session. It contains CSV logs, resolved configuration, provenance, `checkpoints/best.pth` and `checkpoints/last.pt`.

The first bootstrap verifies all LMDB checksums without copying LMDB data from Input. Training reads only the selected QP. GPU execution of this notebook has not been verified locally.
